# Step 5b — LLM zero-shot ASP-relevance scoring (DeepSeek API)

Same goal as the Qwen2.5-7B notebook but uses the DeepSeek API (`deepseek-v4-flash`) instead of a local LLM.

**Why DeepSeek API path:**
- No GPU needed (Colab free CPU runtime is enough).
- Fast: ~5-10 min for 3090 papers with concurrent requests.
- Cheap: 3090 calls × ~600 input tokens ≈ 1.85M input tokens. At DeepSeek-flash pricing this is $0.50-2.
- API supports OpenAI-compatible `logprobs=True, top_logprobs=20`, which lets us recover the continuous score via the *verbalizer* trick (expected value over digit-token probabilities) — cleaner than parsing text output.

**Pipeline:**
1. Configure OpenAI SDK with DeepSeek base URL.
2. Test the prompt + logprobs on 5 sample rows.
3. If logprobs include digit tokens '1'..'5' → use verbalizer (continuous score).
4. Otherwise fallback: parse the first non-whitespace char of the response.
5. Async scoring with concurrency=10 and resume cache.
6. Save in same format as the Qwen / SPECTER2 anchors so the local stacking script picks it up.

**You will need:** A DeepSeek API key (set on https://platform.deepseek.com).

## 1. Install + setup

In [ ]:
!pip install -q --upgrade openai pandas "numpy<2" scipy scikit-learn nest_asyncio

**Note:** if you ever see `ImportError: cannot import name '_center' from 'numpy._core.umath'`, Colab upgraded numpy to 2.x. Run this cell once to restart the runtime cleanly:

```python
!pip install -q "numpy<2" --force-reinstall
import os; os.kill(os.getpid(), 9)
```

After Colab restarts, re-run cell 1, the data-load cells, then load the resume cache before continuing:

```python
cache = pd.read_csv(RUN_DIR / 'llm_raw_cache.csv')
train_res = cache[cache['source_split']=='train'].copy()
public_res = cache[cache['source_split']=='public_test'].copy()
private_res = cache[cache['source_split']=='private_test'].copy()
```

In [ ]:
import os, getpass, pathlib, json, time, asyncio
import pandas as pd, numpy as np
import nest_asyncio
nest_asyncio.apply()  # so asyncio.run() works inside Jupyter

WORK = pathlib.Path('/content/work') if pathlib.Path('/content').exists() else pathlib.Path('./work')
DATA = WORK / 'data'
OUT = WORK / 'outputs'
RUN_DIR = OUT / 'llm_zeroshot'
for d in [WORK, DATA, OUT, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEEPSEEK_API_KEY = os.environ.get('DEEPSEEK_API_KEY') or getpass.getpass('DEEPSEEK_API_KEY: ')
BASE_URL = 'https://api.deepseek.com'
MODEL = 'deepseek-v4-flash'   # alternative: 'deepseek-v4-pro'
print('model =', MODEL, 'base_url =', BASE_URL)

## 2. Upload data + load

In [ ]:
import zipfile
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    uploaded = colab_files.upload()
    for name, content in uploaded.items():
        target = WORK / name
        target.write_bytes(content)
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(target) as zf:
                zf.extractall(DATA)
            print('Extracted', name, '->', DATA)
else:
    print('Local run: place asp_data.zip in', WORK, 'or copy data files into', DATA)
for p in sorted(DATA.glob('*')):
    print(p.name, p.stat().st_size)

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
public = pd.read_csv(DATA / 'public_test.csv')
private = pd.read_csv(DATA / 'private_test.csv')
sample = pd.read_csv(DATA / 'Test_Submission.csv')
abstracts_file = DATA / 'abstracts_merged_v2.csv'
if not abstracts_file.exists():
    abstracts_file = DATA / 'abstracts_merged_v3.csv'
abstracts = pd.read_csv(abstracts_file)
print('using abstract cache:', abstracts_file.name)

abs_map = abstracts[['source_split', 'id', 'abstract', 'has_abstract']]

def attach(df, split):
    df = df.copy()
    df['source_split'] = split
    out = df.merge(abs_map, on=['source_split', 'id'], how='left')
    out['abstract'] = out['abstract'].fillna('')
    out['has_abstract'] = out['has_abstract'].fillna(False).astype(bool)
    return out

train_full = attach(train, 'train').reset_index(drop=True)
public_full = attach(public, 'public_test').reset_index(drop=True)
private_full = attach(private, 'private_test').reset_index(drop=True)
for name, df in [('train', train_full), ('public', public_full), ('private', private_full)]:
    print(f'{name}: rows={len(df)}, has_abstract={int(df["has_abstract"].sum())} ({df["has_abstract"].mean():.1%})')

## 3. Prompt design (same rubric as Qwen path)

In [ ]:
SYSTEM = """You are a research assistant specialized in classifying scientific papers by their relevance to Answer Set Programming (ASP) and the broader AI-symbolic agenda (neuro-symbolic AI, neural-network verification, explainable AI built on logic).

Given a paper's title and abstract, you score its ASP / AI-symbolic relevance on an integer scale 1-5.

Scoring rubric:
- 1 = Not related to ASP. Proceedings volumes, workshop summaries, generic verification / type theory / theoretical-CS papers without an ASP component.
- 2 = Loose adjacency to ASP / KR. Argumentation, description logic, default reasoning, planning, formal verification with no ASP.
- 3 = Generic logic / declarative reasoning that touches ASP vocabulary (Prolog, Datalog, abductive reasoning, NMR, qualitative reasoning) but is not ASP-centric.
- 4 = Applied / extending ASP. Papers that use ASP to solve a domain problem, extend ASP with new constructs (probabilities, choice, preferences, learning), or build tools / encodings on top of an ASP solver.
- 5 = Core ASP advances or ASP × AI cross-overs. ASP solver / grounder algorithms (Clingo, DLV, ASP(Q)), formal semantics of ASP, ASP-driven learning of programs / heuristics, neuro-symbolic systems built on ASP. Inside CAV / LICS, label 5 also covers neural-network verification and ML-meets-formal-methods work.

Examples:

Title: Proceedings 41st International Conference on Logic Programming, ICLP 2025, Rende, Italy.
Abstract: (no abstract available)
Answer: 1

Title: Synthesizing Reactive Systems from Hyperproperties.
Abstract: We present an algorithm for synthesizing reactive systems from hyperproperty specifications. We focus on a fragment of HyperLTL and provide a synthesis algorithm based on bounded synthesis.
Answer: 2

Title: DatalogMTL over the Integer Timeline.
Abstract: We study DatalogMTL, an extension of Datalog with metric temporal operators. Our main contribution is a tight characterisation of the data complexity of reasoning in the integer timeline setting.
Answer: 3

Title: An Answer Set Programming Approach to Argumentative Reasoning in the ASPIC+ Framework.
Abstract: We present an ASP encoding of argumentative reasoning in ASPIC+. The encoding is correct and complete, and we evaluate it on benchmark instances.
Answer: 4

Title: Formally Explaining Decision Tree Models with Answer Set Programming.
Abstract: We propose a formal framework for explaining decision tree predictions using Answer Set Programming. We provide both abductive and contrastive explanations and demonstrate the approach on standard benchmarks.
Answer: 5

Output ONLY a single digit 1, 2, 3, 4, or 5. Do not add any explanation, prefix, or suffix."""

def build_user_message(title, abstract):
    title = (str(title) if title is not None else '').strip()
    abstract = (str(abstract) if abstract is not None else '').strip()
    if len(abstract) > 1500:
        abstract = abstract[:1500] + '...'
    if not abstract:
        abstract = '(no abstract available)'
    return f'Title: {title}\nAbstract: {abstract}\nAnswer:'

def build_messages(title, abstract):
    return [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': build_user_message(title, abstract)},
    ]

print('prompt template ready')

## 4. Sanity test on 3 sample rows + check logprobs support

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=BASE_URL)

samples = train_full.sample(3, random_state=42)
for _, r in samples.iterrows():
    msgs = build_messages(r['title'], r['abstract'])
    resp = client.chat.completions.create(
        model=MODEL,
        messages=msgs,
        max_tokens=4,
        temperature=0.0,
        logprobs=True,
        top_logprobs=10,
    )
    msg = resp.choices[0].message
    text_out = (msg.content or '').strip()
    lp = resp.choices[0].logprobs
    print(f'true={int(r["Label"])}  text={text_out!r}')
    if lp and lp.content:
        first = lp.content[0]
        print('  top logprobs at first token:', [(t.token, round(t.logprob, 3)) for t in first.top_logprobs])
    else:
        print('  WARNING: no logprobs returned')
    print('  title:', str(r['title'])[:80])
    print()

If the top logprobs at the first token include the digits '1'..'5', we can compute a continuous score from them. Otherwise we fall back to text-parse + a heuristic score (the parsed digit, exactly).

## 5. Async scoring with retry + resume cache

In [ ]:
import math, re, traceback
from openai import AsyncOpenAI

CACHE_PATH = RUN_DIR / 'llm_raw_cache.csv'
CONCURRENCY = 10
MAX_RETRIES = 5

DIGIT_TOKENS = {'1', '2', '3', '4', '5',
                ' 1', ' 2', ' 3', ' 4', ' 5',
                '1.', '2.', '3.', '4.', '5.'}
DIGIT_TO_VAL = {'1': 1, '2': 2, '3': 3, '4': 4, '5': 5}

async_client = AsyncOpenAI(api_key=DEEPSEEK_API_KEY, base_url=BASE_URL)

def parse_logprobs_to_score(lp):
    """Return (continuous_score, prob_array_of_5, source) or None if not parseable."""
    if lp is None or not lp.content:
        return None
    first = lp.content[0]
    if not first.top_logprobs:
        return None
    # Collect log-probs for each digit 1..5 (first match across token variants)
    digit_logprobs = {d: None for d in '12345'}
    for t in first.top_logprobs:
        tok = t.token.strip()
        if tok in DIGIT_TO_VAL:
            d = tok
            if digit_logprobs[d] is None or t.logprob > digit_logprobs[d]:
                digit_logprobs[d] = t.logprob
    if all(v is None for v in digit_logprobs.values()):
        return None
    # Substitute very low logprob for missing digits so they don't dominate softmax
    arr = []
    for d in '12345':
        v = digit_logprobs[d]
        arr.append(v if v is not None else -50.0)
    arr = np.array(arr, dtype=np.float64)
    # Softmax over the 5 digits
    arr = arr - arr.max()
    p = np.exp(arr)
    p = p / p.sum()
    score = float(np.dot(p, np.array([1, 2, 3, 4, 5], dtype=float)))
    return score, p.astype(np.float32), 'logprobs'

def parse_text_to_score(text):
    text = (text or '').strip()
    m = re.search(r'[1-5]', text)
    if m:
        d = int(m.group(0))
        # Build a 1-hot probability for fallback diversity calc
        p = np.zeros(5, dtype=np.float32)
        p[d - 1] = 1.0
        return float(d), p, 'text'
    return None

async def score_one(row, semaphore):
    msgs = build_messages(row['title'], row['abstract'])
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            async with semaphore:
                resp = await async_client.chat.completions.create(
                    model=MODEL,
                    messages=msgs,
                    max_tokens=4,
                    temperature=0.0,
                    logprobs=True,
                    top_logprobs=10,
                )
            text_out = (resp.choices[0].message.content or '').strip()
            res = parse_logprobs_to_score(resp.choices[0].logprobs)
            if res is None:
                res = parse_text_to_score(text_out)
            if res is None:
                # fallback to score 3 with uniform probs
                p = np.full(5, 0.2, dtype=np.float32)
                res = (3.0, p, 'fallback')
            score, probs, source = res
            return {'score': score,
                    'p1': float(probs[0]), 'p2': float(probs[1]), 'p3': float(probs[2]),
                    'p4': float(probs[3]), 'p5': float(probs[4]),
                    'source': source, 'text_out': text_out, 'error': ''}
        except Exception as exc:
            last_err = repr(exc)
            await asyncio.sleep(2 ** attempt)
    return {'score': 3.0, 'p1': 0.2, 'p2': 0.2, 'p3': 0.2, 'p4': 0.2, 'p5': 0.2,
            'source': 'error', 'text_out': '', 'error': last_err}

async def score_dataset_async(df, split_name, flush_every=100):
    """Resume-friendly: skips rows already in cache."""
    if CACHE_PATH.exists():
        cache = pd.read_csv(CACHE_PATH)
        done_keys = set(zip(cache['source_split'].astype(str), cache['id'].astype(int)))
    else:
        cache = pd.DataFrame()
        done_keys = set()

    targets = df.copy()
    targets['source_split'] = split_name
    pending = targets[~targets.apply(lambda r: (split_name, int(r['id'])) in done_keys, axis=1)]
    print(f'{split_name}: cached={sum(1 for k in done_keys if k[0]==split_name)} pending={len(pending)}')
    if len(pending) == 0:
        return cache[cache['source_split']==split_name].copy()

    sem = asyncio.Semaphore(CONCURRENCY)
    rows_buffer = []
    t0 = time.time()
    coros = [score_one(r, sem) for _, r in pending.iterrows()]
    for i in range(0, len(coros), flush_every):
        results = await asyncio.gather(*coros[i:i+flush_every])
        for (_, r), res in zip(list(pending.iloc[i:i+flush_every].iterrows()), results):
            rows_buffer.append({
                'source_split': split_name,
                'id': int(r['id']),
                'title': str(r['title']),
                **res,
            })
        partial = pd.DataFrame(rows_buffer)
        merged = pd.concat([cache, partial], ignore_index=True) if len(cache) else partial
        merged = merged.drop_duplicates(['source_split','id'], keep='last')
        merged.to_csv(CACHE_PATH, index=False, encoding='utf-8-sig')
        rate = (i + flush_every) / max(time.time() - t0, 1e-6)
        print(f'  {split_name}: flushed {min(i+flush_every, len(coros))}/{len(coros)}  rate={rate:.1f} req/s  errors={sum(1 for x in rows_buffer if x["source"]=="error")}')
    return pd.read_csv(CACHE_PATH).query('source_split == @split_name').copy()

async def run_all():
    train_res = await score_dataset_async(train_full, 'train')
    public_res = await score_dataset_async(public_full, 'public_test')
    private_res = await score_dataset_async(private_full, 'private_test')
    return train_res, public_res, private_res

train_res, public_res, private_res = asyncio.run(run_all())
print('done. cached rows:', sum(len(x) for x in [train_res, public_res, private_res]))
print('source breakdown:')
print(pd.concat([train_res, public_res, private_res]).groupby('source').size())

## 6. Sanity check + threshold tuning

In [ ]:
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, f1_score
from scipy.optimize import differential_evolution

train_res = train_res.sort_values('id').reset_index(drop=True)
public_res = public_res.sort_values('id').reset_index(drop=True)
private_res = private_res.sort_values('id').reset_index(drop=True)

# Align order with train_full / public_full / private_full by id
train_full_sorted = train_full.sort_values('id').reset_index(drop=True)
public_full_sorted = public_full.sort_values('id').reset_index(drop=True)
private_full_sorted = private_full.sort_values('id').reset_index(drop=True)

# Make sure ids align
assert (train_res['id'].to_numpy() == train_full_sorted['id'].to_numpy()).all()
assert (public_res['id'].to_numpy() == public_full_sorted['id'].to_numpy()).all()
assert (private_res['id'].to_numpy() == private_full_sorted['id'].to_numpy()).all()

y_class = train_full_sorted['Label'].astype(int).to_numpy()
train_scores = train_res['score'].to_numpy()
public_scores = public_res['score'].to_numpy()
private_scores = private_res['score'].to_numpy()
train_probs = train_res[['p1','p2','p3','p4','p5']].to_numpy()
public_probs = public_res[['p1','p2','p3','p4','p5']].to_numpy()
private_probs = private_res[['p1','p2','p3','p4','p5']].to_numpy()

_per_label = pd.DataFrame({'true': y_class, 'score': train_scores}).groupby('true')['score'].agg(['mean','std','count'])
print('Mean LLM score per true label (train):')
print(_per_label.round(3).to_string())
rounded = np.clip(np.round(train_scores), 1, 5).astype(int)
print('\nNo-tuning round-QWK on train:', round(cohen_kappa_score(y_class, rounded, weights='quadratic'), 4))
print('No-tuning MAE:', round(mean_absolute_error(y_class, rounded), 4))

In [ ]:
TRAIN_DIST = pd.Series(y_class).value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=0).to_numpy()
DIST_PENALTY_LAMBDA = 0.5

def scores_to_labels(scores, thresholds):
    return np.digitize(scores, np.sort(np.asarray(thresholds, dtype=float))) + 1

def predicted_dist(labels):
    return pd.Series(labels).value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=0).to_numpy()

def tune_thresholds_constrained(y_true, scores, lambd=DIST_PENALTY_LAMBDA, seed=42):
    def objective(raw):
        thr = np.sort(raw)
        gap = np.min(np.diff(thr))
        gap_pen = 0.0 if gap >= 0.03 else (0.03 - gap) * 5.0
        labels = scores_to_labels(scores, thr)
        qwk = cohen_kappa_score(y_true, labels, weights='quadratic')
        dist_pen = float(np.sum(np.abs(predicted_dist(labels) - TRAIN_DIST)))
        return -qwk + gap_pen + lambd * dist_pen
    bounds = [(1.4, 2.5), (1.8, 2.9), (2.2, 3.4), (2.6, 4.2)]
    res = differential_evolution(objective, bounds, seed=seed, maxiter=120, popsize=15,
                                 polish=True, updating='immediate', workers=1)
    thr = np.sort(res.x)
    return thr, cohen_kappa_score(y_true, scores_to_labels(scores, thr), weights='quadratic')

thresholds, oof_qwk = tune_thresholds_constrained(y_class, train_scores)
oof_pred = scores_to_labels(train_scores, thresholds)
print('Constrained-tuned OOF QWK =', round(oof_qwk, 4))
print('thresholds =', thresholds.tolist())
print('OOF predicted dist =', dict(zip([1,2,3,4,5], predicted_dist(oof_pred).round(3).tolist())))
print('TRAIN actual dist  =', dict(zip([1,2,3,4,5], TRAIN_DIST.round(3).tolist())))
print('OOF MAE =', round(mean_absolute_error(y_class, oof_pred), 4))
print('OOF macro-F1 =', round(f1_score(y_class, oof_pred, average='macro'), 4))

## 7. Save artefacts

In [ ]:
public_pred = scores_to_labels(public_scores, thresholds)
private_pred = scores_to_labels(private_scores, thresholds)

metrics = {
    'method': 'llm_zeroshot_deepseek',
    'model': MODEL,
    'base_url': BASE_URL,
    'concurrency': CONCURRENCY,
    'oof_qwk': float(oof_qwk),
    'oof_mae': float(mean_absolute_error(y_class, oof_pred)),
    'oof_macro_f1': float(f1_score(y_class, oof_pred, average='macro')),
    'thresholds': [float(v) for v in thresholds],
    'label_distribution_combined': {int(k): int(v) for k, v in pd.Series(
        np.concatenate([public_pred, private_pred])).value_counts().sort_index().items()},
    'label_distribution_public': {int(k): int(v) for k, v in pd.Series(public_pred).value_counts().sort_index().items()},
    'label_distribution_private': {int(k): int(v) for k, v in pd.Series(private_pred).value_counts().sort_index().items()},
    'note': 'Continuous score = sum_d d * P(token=d) for d in 1..5 from logprobs. Pure zero-shot, no fold-based fitting; OOF == train scoring directly.',
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

pd.DataFrame({'id': train_full_sorted['id'], 'Label': y_class,
              'oof_score': train_scores, 'oof_pred': oof_pred,
              **{f'p_label_{i+1}': train_probs[:, i] for i in range(5)}}).to_csv(
    RUN_DIR / 'oof_scores.csv', index=False)
pd.DataFrame({'id': public_full_sorted['id'], 'score': public_scores, 'pred': public_pred,
              **{f'p_label_{i+1}': public_probs[:, i] for i in range(5)}}).to_csv(
    RUN_DIR / 'public_scores.csv', index=False)
pd.DataFrame({'id': private_full_sorted['id'], 'score': private_scores, 'pred': private_pred,
              **{f'p_label_{i+1}': private_probs[:, i] for i in range(5)}}).to_csv(
    RUN_DIR / 'private_scores.csv', index=False)

combo = pd.concat([
    pd.DataFrame({'id': public_full_sorted['id'], 'Label': public_pred}),
    pd.DataFrame({'id': private_full_sorted['id'], 'Label': private_pred}),
], ignore_index=True)
submission = sample[['id']].merge(combo, on='id', how='left')
submission['Label'] = submission['Label'].astype(int)
submission.to_csv(RUN_DIR / 'llm_zeroshot_submission.csv', index=False)
print('submission rows =', len(submission))
print(submission.head())

## 8. Zip + download (Colab) or just inspect locally

In [ ]:
zip_path = pathlib.Path('/content/llm_zeroshot_outputs.zip') if IN_COLAB else (WORK / 'llm_zeroshot_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.iterdir():
        zf.write(p, arcname=f'llm_zeroshot/{p.name}')
print('zipped:', zip_path, 'size MB =', round(zip_path.stat().st_size / 1e6, 2))
if IN_COLAB:
    colab_files.download(str(zip_path))